# Exploring Pretrained Models with Hugging Face

**Optional Week 09 extension · 15–20 minutes · ungraded**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module9/week09_huggingface_intro.ipynb)

In the LSTM practice you trained a small classifier. Here you reuse a model whose weights have already been learned. You will load a sentiment pipeline, inspect its tokens, and investigate negation and contrast. No training is needed.

Hugging Face provides a **Hub** of shared models and the **Transformers** Python library for using them. A checkpoint contains learned weights and configuration. Our checkpoint is DistilBERT, a smaller BERT-style transformer fine-tuned on SST-2 English movie-review sentiment. It predicts **NEGATIVE** or **POSITIVE**; it is not a text-generating chatbot.

By the end, you should be able to trace a pipeline from text to scores and explain why a confident prediction can still be misleading.

## 1. Load a ready-to-use classifier (3 minutes)

Choose a standard **CPU** runtime in Colab. Run the cells from top to bottom. The first run needs internet to download the public model (roughly 270 MB); later runs can use its cache. No Hugging Face account, token, or API key is required.

Install the tested Transformers version with `%pip`, which targets this notebook's Python environment. Colab supplies PyTorch. If Colab requests a runtime restart, restart and run all cells again.

In [1]:
%pip -q install "transformers==5.17.0"

In [2]:
import torch
import transformers
from transformers import pipeline

print("Transformers:", transformers.__version__, "| PyTorch:", torch.__version__)
torch.set_num_threads(2)
MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline(
    "sentiment-analysis",
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    device=-1,  # CPU
    model_kwargs={"disable_mmap": True},  # Load weights into memory for CPU compatibility
)
review = "Everyone told me the film was boring, but I loved it."
classifier(review)

Transformers: 5.17.0 | PyTorch: 2.14.0


[{'label': 'POSITIVE', 'score': 0.9997287392616272}]

The **label** is the class with the highest score. For this two-class model, the **score** is that class's softmax probability. It is not a rating out of five, a measure of how emotional the writer is, or a guarantee that the prediction is correct.

The checkpoint is named explicitly so we know which model and tokenizer we are using. Its parameters stay fixed during this activity.

**Question 1:** Before running the next cell, predict the labels for “The film was good” and “The film was not good.” What would count as evidence that negation changed the prediction?

In [3]:
examples = ["The film was good.", "The film was not good."]
for text, result in zip(examples, classifier(examples)):
    print(f"{result['label']:8s} {result['score']:.4f} | {text}")

POSITIVE 0.9999 | The film was good.
NEGATIVE 0.9997 | The film was not good.


## 2. Look behind the pipeline (5 minutes)

A pipeline joins three steps:

1. **Preprocess:** the tokenizer splits text into tokens and maps them to vocabulary IDs.
2. **Run the model:** embeddings and transformer layers produce representations; a classification head produces two raw scores, called logits.
3. **Postprocess:** softmax converts logits into probabilities, and the pipeline attaches label names.

Use the tokenizer already attached to our classifier. Its vocabulary and preprocessing must match the model. An ID is a lookup index, not a sentiment score or an embedding.

In [4]:
tokenizer = classifier.tokenizer
encoded = tokenizer(review)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print("Token           ID")
for token, token_id in zip(tokens, encoded["input_ids"]):
    print(f"{token:14s} {token_id}")
print("Attention mask:", encoded["attention_mask"])
print("Subword example:", tokenizer.tokenize("unbelievably watchable"))

Token           ID
[CLS]          101
everyone       3071
told           2409
me             2033
the            1996
film           2143
was            2001
boring         11771
,              1010
but            2021
i              1045
loved          3866
it             2009
.              1012
[SEP]          102
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Subword example: ['un', '##bel', '##ie', '##va', '##bly', 'watch', '##able']


`[CLS]` and `[SEP]` are special boundary tokens. This uncased tokenizer lowercases the text; punctuation can have its own token. A word may split into smaller pieces: `##` marks a continuation in this tokenizer. Tokenization can differ from the simple word splitting used in the LSTM practice.

The attention mask marks real tokens with 1 and padding with 0. There is no padding in this single example. It is not an explanation of which words mattered most.

**Question 2:** Which tokens differ from splitting the original review on spaces? Why would using another model's token IDs be a problem?

Now expose the model and postprocessing steps without loading another model. `return_tensors="pt"` creates the PyTorch tensors you met in the LSTM practice. Inference mode avoids tracking gradients; no weights are updated.

In [5]:
inputs = tokenizer(review, return_tensors="pt", truncation=True)
classifier.model.eval()
with torch.inference_mode():
    logits = classifier.model(**inputs).logits
probabilities = logits.softmax(dim=-1)[0]
print("Input shape:", tuple(inputs["input_ids"].shape))
print("Raw logits:", logits.tolist())
for label_id, probability in enumerate(probabilities):
    print(classifier.model.config.id2label[label_id], f"{probability.item():.4f}")

# Check that these exposed steps reproduce pipeline postprocessing.
result = classifier(review)[0]
best_id = int(probabilities.argmax())
assert result["label"] == classifier.model.config.id2label[best_id]
assert abs(result["score"] - probabilities[best_id].item()) < 1e-5
assert torch.isclose(probabilities.sum(), torch.tensor(1.0))

Input shape: (1, 15)
Raw logits: [[-3.994314193725586, 4.217988014221191]]
NEGATIVE 0.0003
POSITIVE 0.9997


## 3. Try negation and contrast (5–7 minutes)

Predict what will happen before running the cell. The first pair uses the same words in a different order. The next pair reverses the clauses of our running review. The final example mixes opinions without a clear positive or negative verdict.

`top_k=None` asks for both labels. We select probabilities by label name rather than assuming the returned list has a particular order.

In [6]:
reviews = [
    "The acting was good, not bad.",
    "The acting was bad, not good.",
    "Everyone told me the film was boring, but I loved it.",
    "I loved it, but everyone told me the film was boring.",
    "The film was neither good nor bad.",
]
all_scores = classifier(reviews, top_k=None, truncation=True)
print("P(positive)  P(negative)  Predicted label | Review")
for text, scores in zip(reviews, all_scores):
    by_label = {item["label"]: item["score"] for item in scores}
    winner = max(scores, key=lambda item: item["score"])["label"]
    print(f"{by_label['POSITIVE']:.4f}       {by_label['NEGATIVE']:.4f}       {winner:8s} | {text}")
    assert abs(sum(by_label.values()) - 1.0) < 1e-5

P(positive)  P(negative)  Predicted label | Review
0.9997       0.0003       POSITIVE | The acting was good, not bad.
0.0002       0.9998       NEGATIVE | The acting was bad, not good.
0.9997       0.0003       POSITIVE | Everyone told me the film was boring, but I loved it.
0.0017       0.9983       NEGATIVE | I loved it, but everyone told me the film was boring.
0.0132       0.9868       NEGATIVE | The film was neither good nor bad.


**Question 3:** Choose one prediction you find questionable. Explain your interpretation of the review and why even a high model score would not settle the disagreement. Then change one phrase below and see whether the prediction changes.

In [7]:
my_review = "The film was boring, but the ending was wonderful."
classifier(my_review, top_k=None, truncation=True)

[{'label': 'POSITIVE', 'score': 0.9997474551200867},
 {'label': 'NEGATIVE', 'score': 0.0002525213931221515}]

## 4. Interpret the result (2–3 minutes)

A change in these examples shows sensitivity to that wording, not reliable language understanding. Reversing clauses can change emphasis without changing who holds an opinion. Human readers may also disagree on mixed reviews.

This classifier must choose between two labels: it has no neutral or mixed class. Its scores sum to one even for unclear or unfamiliar input. Training data can encode biases, and performance on English movie reviews does not establish performance on other languages, topics, sarcasm, or groups of people. A high softmax score is not a calibrated guarantee of accuracy. Long inputs may be truncated, hiding a later qualification. Assess reliability with representative labeled test data, not just these few examples.

**Connecting to Week 10:** the LSTM processed tokens recurrently. DistilBERT uses transformer attention to form contextual representations, then a classifier predicts sentiment. Reusing a pretrained representation for a task is transfer learning. We used an already fine-tuned checkpoint; we did not fine-tune it ourselves. Week 10 will explain attention and how encoder models such as BERT differ from generative LLMs.

### Official references

- [Hugging Face: Working with pipelines](https://huggingface.co/learn/llm-course/en/chapter1/3)
- [Hugging Face: Behind the pipeline](https://huggingface.co/learn/llm-course/en/chapter2/2)
- [Checkpoint model card, training details, and limitations](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english)

Explanations and examples here are adapted for Week 09; the output above comes from your execution.